In [1]:
from data_gen import make_loans
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

In [2]:
df , approved = make_loans()

print("rows:", len(df), "| approved share:", round(approved.mean(), 2))
print(df.head())

rows: 800 | approved share: 0.6
       income  loan_amount  age credit_history    purpose
0   82.509547    59.847515   53           good       home
1  109.721380    13.616798   54           good  education
2   97.568569    49.707048   48           none        car
3   42.520719    27.610114   43           poor  education
4   50.016628    48.081914   58           poor        car


In [3]:
df["dti"] = df["loan_amount"]/df["income"]

In [4]:
df["dti"]
df

,income,loan_amount,age,credit_history,purpose,dti
0,82.509547,59.847515,53,good,home,0.725340
1,109.721380,13.616798,54,good,education,0.124103
2,97.568569,49.707048,48,none,car,0.509458
3,42.520719,27.610114,43,poor,education,0.649333
4,50.016628,48.081914,58,poor,car,0.961319
...,...,...,...,...,...,...
795,76.859173,52.018187,58,good,education,0.676799
796,65.879072,39.596722,40,poor,car,0.601052
797,76.398455,35.742244,26,good,education,0.467840
798,93.720354,17.243964,54,good,business,0.183994


In [7]:
categorical = df.select_dtypes(include = ['object' , 'category']).columns
numeric = df.select_dtypes(exclude = ['object' , 'category']).columns

In [8]:
crf = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown="ignore"), categorical)
])

models = {
    "Logistic Regression": Pipeline([
        ('crf', crf),
        ('clf', LogisticRegression(max_iter=1000))
    ]),

    "Decision Tree": Pipeline([
        ('crf', crf),
        ('clf', DecisionTreeClassifier(max_depth=4))
    ]),

    "Random Forest": Pipeline([
        ('crf', crf),
        ('clf', RandomForestClassifier(n_estimators=300, random_state=42))
    ])
}

In [9]:
for name , m in models.items():
    score = cross_val_score(m , df , approved , cv = 2)
    print(f"{round(score.mean(),3)} , {round(score.std(),3)}")

0.944 , 0.011
0.924 , 0.004
0.942 , 0.005
